# 💳 Credit Card Fraud Detection — End-to-End Project

This notebook builds a complete, deployable fraud detection pipeline:

- Exploratory Data Analysis (EDA)
- Proper preprocessing (no data leakage)
- Class imbalance handling with **SMOTE** (applied only to training data)
- Multiple models trained & compared (Logistic Regression, Random Forest)
- Full evaluation suite (Precision, Recall, F1, ROC-AUC, Confusion Matrix)
- Saved model artifacts ready to plug into a **Streamlit** web app

**Dataset:** Kaggle "Credit Card Fraud Detection" dataset (`creditcard.csv`) — 284,807 transactions, 30 features (`Time`, `V1`–`V28` PCA components, `Amount`) plus the target `Class` (0 = legit, 1 = fraud).

> Run all cells top to bottom in Google Colab. At the end you'll download 4 files needed for the Streamlit app.

## 1. Setup & Installs

In [ ]:
!pip install -q imbalanced-learn

## 2. Import Dependencies

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    precision_recall_curve, roc_curve, ConfusionMatrixDisplay
)
from imblearn.over_sampling import SMOTE

sns.set_style('whitegrid')
%matplotlib inline

RANDOM_STATE = 42

## 3. Load the Dataset

Upload `creditcard.csv` when prompted (or place it in `/content/` beforehand, e.g. via Google Drive).

In [ ]:
from google.colab import files
import os

DATA_PATH = '/content/creditcard.csv'

if not os.path.exists(DATA_PATH):
    print("Dataset not found at", DATA_PATH)
    print("Please upload creditcard.csv now...")
    uploaded = files.upload()
    DATA_PATH = list(uploaded.keys())[0]

credit_card_data = pd.read_csv(DATA_PATH)
print("Dataset loaded successfully:", credit_card_data.shape)

## 4. Initial Exploration

In [ ]:
credit_card_data.head()

In [ ]:
credit_card_data.info()

In [ ]:
# check for missing values
credit_card_data.isnull().sum().sum()

In [ ]:
credit_card_data.describe()

### Class Distribution

This dataset is **highly imbalanced** — fraud cases are a tiny fraction of all transactions.

In [ ]:
class_counts = credit_card_data['Class'].value_counts()
fraud_pct = class_counts[1] / class_counts.sum() * 100

print(class_counts)
print(f"\nFraudulent transactions: {fraud_pct:.4f}% of all transactions")

plt.figure(figsize=(5,4))
sns.barplot(x=class_counts.index, y=class_counts.values, palette=['#2ecc71', '#e74c3c'])
plt.xticks([0,1], ['Legit (0)', 'Fraud (1)'])
plt.ylabel('Count')
plt.title('Class Distribution')
plt.yscale('log')
plt.show()

### Amount & Time Distributions by Class

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for cls, label, color in [(0, 'Legit', '#2ecc71'), (1, 'Fraud', '#e74c3c')]:
    subset = credit_card_data[credit_card_data['Class'] == cls]
    axes[0].hist(subset['Amount'], bins=50, alpha=0.6, label=label, color=color, density=True)
    axes[1].hist(subset['Time'], bins=50, alpha=0.6, label=label, color=color, density=True)

axes[0].set_title('Transaction Amount Distribution')
axes[0].set_xlabel('Amount'); axes[0].set_xlim(0, 2000); axes[0].legend()

axes[1].set_title('Transaction Time Distribution (seconds since first transaction)')
axes[1].set_xlabel('Time'); axes[1].legend()

plt.tight_layout()
plt.show()

### Correlation Heatmap

In [ ]:
plt.figure(figsize=(16, 12))
corr = credit_card_data.corr()
sns.heatmap(corr, cmap='coolwarm', center=0, cbar_kws={'shrink': 0.6})
plt.title('Feature Correlation Heatmap')
plt.show()

## 5. Feature Scaling

`V1`–`V28` are already PCA-transformed and roughly scaled. `Amount` and `Time` are **not** — we scale them with `RobustScaler` (robust to the extreme outliers common in fraud data).

In [ ]:
scaler_amount = RobustScaler()
scaler_time = RobustScaler()

credit_card_data['scaled_amount'] = scaler_amount.fit_transform(credit_card_data[['Amount']])
credit_card_data['scaled_time'] = scaler_time.fit_transform(credit_card_data[['Time']])

credit_card_data.drop(['Amount', 'Time'], axis=1, inplace=True)

# move scaled columns to the front for readability
scaled_amount = credit_card_data.pop('scaled_amount')
scaled_time = credit_card_data.pop('scaled_time')
credit_card_data.insert(0, 'scaled_amount', scaled_amount)
credit_card_data.insert(1, 'scaled_time', scaled_time)

credit_card_data.head()

## 6. Train/Test Split

**Important:** we split into train/test *before* touching class balance. The test set stays untouched and reflects the real-world imbalance — that's what "accuracy" should be measured against.

In [ ]:
X = credit_card_data.drop(columns='Class', axis=1)
Y = credit_card_data['Class']

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, stratify=Y, random_state=RANDOM_STATE
)

print("Train shape:", X_train.shape, "| Fraud cases in train:", Y_train.sum())
print("Test shape :", X_test.shape, "| Fraud cases in test :", Y_test.sum())

## 7. Handling Class Imbalance with SMOTE

Instead of throwing away ~99% of legit transactions (undersampling), we use **SMOTE** to synthetically oversample the minority (fraud) class — and critically, we apply it **only to the training set**, never the test set.

In [ ]:
smote = SMOTE(random_state=RANDOM_STATE)
X_train_res, Y_train_res = smote.fit_resample(X_train, Y_train)

print("Before SMOTE:", dict(Y_train.value_counts()))
print("After SMOTE :", dict(pd.Series(Y_train_res).value_counts()))

## 8. Model Training & Evaluation

We train two models on the resampled training data and evaluate both on the original, untouched, imbalanced test set — this is what matters for real-world performance.

In [ ]:
def evaluate_model(name, model, X_test, Y_test):
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]

    metrics = {
        'Model': name,
        'Accuracy': accuracy_score(Y_test, preds),
        'Precision': precision_score(Y_test, preds),
        'Recall': recall_score(Y_test, preds),
        'F1 Score': f1_score(Y_test, preds),
        'ROC-AUC': roc_auc_score(Y_test, probs)
    }

    print(f"\n{'='*55}\n{name}\n{'='*55}")
    print(classification_report(Y_test, preds, target_names=['Legit', 'Fraud']))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    ConfusionMatrixDisplay.from_predictions(
        Y_test, preds, display_labels=['Legit', 'Fraud'], ax=axes[0], cmap='Blues'
    )
    axes[0].set_title(f'{name} — Confusion Matrix')

    fpr, tpr, _ = roc_curve(Y_test, probs)
    axes[1].plot(fpr, tpr, label=f'AUC = {metrics["ROC-AUC"]:.4f}', color='#3498db')
    axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5)
    axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title(f'{name} — ROC Curve'); axes[1].legend()
    plt.tight_layout()
    plt.show()

    return metrics

In [ ]:
log_reg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
log_reg.fit(X_train_res, Y_train_res)
lr_metrics = evaluate_model("Logistic Regression", log_reg, X_test, Y_test)

In [ ]:
rf_clf = RandomForestClassifier(
    n_estimators=150, max_depth=12, random_state=RANDOM_STATE, n_jobs=-1
)
rf_clf.fit(X_train_res, Y_train_res)
rf_metrics = evaluate_model("Random Forest", rf_clf, X_test, Y_test)

## 9. Model Comparison

In [ ]:
results_df = pd.DataFrame([lr_metrics, rf_metrics]).set_index('Model')
display(results_df.style.format('{:.4f}').background_gradient(cmap='Greens'))

results_df.drop(columns=[]).plot(kind='bar', figsize=(10, 5), rot=0)
plt.title('Model Comparison')
plt.ylabel('Score')
plt.legend(loc='lower right')
plt.ylim(0, 1.05)
plt.show()

## 10. Feature Importance (Random Forest)

In [ ]:
importances = pd.Series(rf_clf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 6))
sns.barplot(x=importances.head(15).values, y=importances.head(15).index, palette='viridis')
plt.title('Top 15 Most Important Features')
plt.xlabel('Importance')
plt.show()

## 11. Precision–Recall Tradeoff & Threshold Tuning

In fraud detection, missing a fraud (false negative) is usually costlier than flagging a legit transaction for review (false positive). The default 0.5 probability threshold isn't always optimal — this plot helps you pick a better one.

In [ ]:
best_model = rf_clf  # swap to log_reg if it performs better on your run
best_name = "Random Forest"

probs = best_model.predict_proba(X_test)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(Y_test, probs)

plt.figure(figsize=(8, 5))
plt.plot(thresholds, precisions[:-1], label='Precision')
plt.plot(thresholds, recalls[:-1], label='Recall')
plt.xlabel('Decision Threshold')
plt.ylabel('Score')
plt.title(f'{best_name} — Precision/Recall vs Threshold')
plt.legend()
plt.show()

print("Adjust the threshold in the Streamlit app based on this curve —")
print("lower threshold = catch more fraud but more false alarms, and vice versa.")

## 12. Save Model Artifacts

These 4 files are everything the Streamlit app needs. Pick whichever model performed best for your run (Random Forest is used below by default).

In [ ]:
final_model = best_model  # the model chosen above

joblib.dump(final_model, 'fraud_detection_model.pkl')
joblib.dump(scaler_amount, 'scaler_amount.pkl')
joblib.dump(scaler_time, 'scaler_time.pkl')
joblib.dump(list(X.columns), 'feature_columns.pkl')

print("Saved artifacts:")
print(" - fraud_detection_model.pkl")
print(" - scaler_amount.pkl")
print(" - scaler_time.pkl")
print(" - feature_columns.pkl")

## 13. Download Artifacts (for the Streamlit App)

In [ ]:
from google.colab import files

for f in ['fraud_detection_model.pkl', 'scaler_amount.pkl', 'scaler_time.pkl', 'feature_columns.pkl']:
    files.download(f)

## 🚀 Next Steps — Deploying to Streamlit

1. Download the 4 `.pkl` files above.
2. Put them in the same folder as `app.py` and `requirements.txt` (provided alongside this notebook).
3. Push that folder to a **public GitHub repo**.
4. Go to [share.streamlit.io](https://share.streamlit.io), connect your GitHub, and deploy the repo (entry point: `app.py`).
5. Done — you'll get a live public URL for your fraud detection app.

See the accompanying `README.md` for the full step-by-step guide.